# ***IMPORTAR LIBRERIAS***

In [1]:
import openpyxl
from openpyxl.worksheet.table import Table, TableStyleInfo
import os

from alive_progress import alive_bar
#import time

# ***FUNCIONES***

In [2]:
def create_table(path, wb_obj):
    
    if "Resultado" in wb_obj.sheetnames:
    
        print("[!] Tabla Resultado creada, bórrela y vuelva a lanzar el código")
    
    else:
        ws1 = wb_obj.create_sheet("Resultado")
        ws1['B2'] = "LA"
        ws1['C2'] = "Descripcion"
        ws1['D2'] = "Precio"
        ws1['E2'] = "Revisado"
        tab = Table(displayName="Resultado", ref="B2:E{}".format(row_limit+1))
        style = TableStyleInfo(name="TableStyleMedium10", showFirstColumn=False,showLastColumn=False, showRowStripes=True, showColumnStripes=False)
        tab.tableStyleInfo = style
        ws1.add_table(tab)
        wb_obj.save(path)
        print("[+] Añadida un nueva Hoja 'Resultado' y una tabla con los datos")
        
def select_item(la, descripcion, la_c, descripcion_c):
    print("[+] LA iguales y Descripcion diferente, por favor elija \n")
    print("[1] LA: {} Descripción: {}".format(la, descripcion))
    print("[2] LA: {} Descripción: {}".format(la_c, descripcion_c))
    respuesta = int(input("\n[?] Seleccione la opción que quiere mantener"))
    return respuesta

def input_values_table(registro,la_col, descripcion_col, precio_col, revisado_col, path, wb_obj):
    sheet_result = wb_obj['Resultado']
    row_limit = sheet_result.max_row
    
    c1 = sheet_result.cell(row = row_limit+1, column = la_col)
    c1.value = registro[0]
    
    c1 = sheet_result.cell(row = row_limit+1, column = descripcion_col)
    c1.value = registro[1]
    
    c1 = sheet_result.cell(row = row_limit+1, column = precio_col)
    c1.value = registro[2]
    
    c1 = sheet_result.cell(row = row_limit+1, column = revisado_col)
    c1.value = registro[3]
    
    wb_obj.save(path)
    

# ***CONFIGURACION ARCHIVO EXCEL***

In [ ]:
#Incluir la localizacion del fichero excel con los datos de Zooplus:
directory = os.getcwd()
path = directory + "\\Filtrado_precios.xlsx"
#print(directory)
with alive_bar(1) as bar:   # default setting
    for i in range(1):
        #time.sleep(0.03)
        # Se abre el workbook y la hoja seleccionados (donde se guardo el documento por ultima vez)
        wb_obj = openpyxl.load_workbook(path)
        sheet_obj = wb_obj.active

        # Obtener el numero de filas y columnas usadas
        row_limit = sheet_obj.max_row
        column_limit = sheet_obj.max_column
        
        #Introducir el numero de columna en la que se encuentra el EAN (A=1, B=2, C=3...)
        la_col = 1

        #Introducir el numero de la fila en la que se encuentra el primer valor de EAN
        first_la_row = 2

        #Introducir el numero de columna en la que se encuentra el precio
        precio_col = 3
        
        #Opciones de revision
        revisado_col = 4
        
        #Descripcion
        descripcion_col = 2
        
        bar()

print("[+] Total de filas:", row_limit)
print("[+] Total de columnas:", column_limit)
create_table(path, wb_obj)


# ***COMIENZO ANALISIS Y PURGA***

In [8]:
print("[+] Comienzo de analisis de precios")
       
#Cogemos el primer LA, descripcion    
for i in range(first_la_row, row_limit + 1): 
            cell_obj = sheet_obj.cell(row = i, column = la_col)
            cell_obj_des = sheet_obj.cell(row = i, column = descripcion_col)
            cell_obj_pr1 = sheet_obj.cell(row = i, column = precio_col)
            precio_la_1 = float(cell_obj_pr1.value)
            
            la = str(cell_obj.value)
            descripcion = str(cell_obj_des.value)
            
            #Recorremos la lista comparando el primer LA con cada uno de los siguientes
            for j in range(first_la_row, row_limit+1):
                cell_obj2 = sheet_obj.cell(row = j, column = la_col)
                cell_obj_des2 = sheet_obj.cell(row = j, column = descripcion_col)
               
                cell_obj_pr2 = sheet_obj.cell(row = j, column = precio_col)
                
                precio_la_2 = float(cell_obj_pr2.value)
                la_c = str(cell_obj2.value)
                descripcion_c= str(cell_obj_des2)
                
                #Comparamos si los LA y descripcion son iguales
                if la == la_c and descripcion == descripcion_c:
                    cell_obj_rev = sheet_obj.cell(row = i, column = revisado_col)
                    revisado = str(cell_obj_rev.value).capitalize()
                    #Si ya se ha revisado, se informa y se pone el precio contrastado
                    if revisado == "X":
                        print("[+] LA {} Revisado por el personal")
                        input_values_table(la, descripcion, precio_la_1, revisado, wb_obj, la_col, descripcion_col, precio_col, revisado_col, path)
                    else:
                        #revisado = ""  
                        maximo = max(precio_la_1, precio_la_2)
                        input_values_table(la, descripcion, maximo, revisado, wb_obj, la_col, descripcion_col, precio_col, revisado_col, path)
                #Si el LA es igual pero la descripcion cambia, debemos elegir cual es el nuevo
                elif la == la_c and descripcion != descripcion_c:
                    while True:
                        respuesta = select_item(la, descripcion, la_c, descripcion_c)
                        if respuesta == 1:
                            input_values_table(la, descripcion, precio_la_1 , revisado, wb_obj, la_col, descripcion_col, precio_col, revisado_col, path)
                            
                            break
                        if respuesta == 2:
                            input_values_table(la_c, descripcion_c, precio_la_2 , revisado, wb_obj, la_col, descripcion_col, precio_col, revisado_col, path)
                            break
                        else:
                            print("\n")
                            print("*"*32)
                            print("[!] Respuesta incorrecta, elija 1 o 2")
                            print("*"*32)
                            print("\n")

wb_obj.save(path)
print("[+] Proceso terminado")                           